# Lily 1.5b VLM GGUF & `mmproj` Export — Google Colab
**Builds `llama.cpp`, converts LLM backbone & SigLIP2 projector to GGUF, quantizes, and uploads**

This notebook compiles `llama.cpp` with multimodal tools, patches `tokenizer_config.json` with the exact GRPO chat template, converts the LLM backbone to F16 GGUF, converts the SigLIP2 vision tower + projector to `mmproj-model-f16.gguf`, executes K-quantizations (`Q4_K_M`, `Q5_K_M`, `Q8_0`), verifies via live streaming CPU smoke tests, and uploads all artifacts to `abhinav0231/Lily-1.5b-VLM-GGUF`.


## Cell 1 — Clone `llama.cpp` & Compile Multimodal Binaries


In [ ]:
# ==============================================================================
# Cell 1 — Clone llama.cpp & Compile Multimodal Binary Tools via CMake
# ==============================================================================
!apt-get update -qq && apt-get install -y cmake build-essential -qq
!git clone https://github.com/ggerganov/llama.cpp.git /content/llama.cpp
%cd /content/llama.cpp
!mkdir -p build && cd build && cmake .. -DGGML_NATIVE=OFF && cmake --build . --config Release -j$(nproc)
%cd /content
!pip install -q huggingface_hub sentencepiece gguf pillow torchvision
print("✅ llama.cpp multimodal binaries built successfully")


## Cell 2 — Download Base VLM & Patch Jinja Chat Template


In [ ]:
# ==============================================================================
# Cell 2 — Download Base VLM & Patch Jinja Chat Template
# ==============================================================================
import os
import json
from huggingface_hub import snapshot_download

MODEL_REPO = "abhinav0231/Lily-1.5b-v0.5-Vision-SFT"
LOCAL_DIR  = "/content/Lily-1.5b-v0.5-Vision-SFT"

print(f"Downloading model snapshot from {MODEL_REPO} ...")
snapshot_download(
    repo_id   = MODEL_REPO,
    local_dir = LOCAL_DIR,
    ignore_patterns = ["*.msgpack", "*.h5", "*.ot"]
)
print(f"✅ Model downloaded to: {LOCAL_DIR}")

# ── Patch tokenizer_config.json with explicit GRPO Chat Template ──────────────
tok_cfg_path = os.path.join(LOCAL_DIR, "tokenizer_config.json")
if os.path.exists(tok_cfg_path):
    with open(tok_cfg_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    SYSTEM_PROMPT = "You are a precise, helpful assistant. Always reason step by step inside <think> tags, then write your final answer inside <answer> tags."

    ROBUST_CHAT_TEMPLATE = (
        "{% if messages[0]['role'] == 'system' %}"
        "{{ '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}"
        "{% else %}"
        "{{ '<|im_start|>system\n" + SYSTEM_PROMPT + "<|im_end|>\n' }}"
        "{% endif %}"
        "{% for message in messages %}"
        "{% if message['role'] == 'user' %}"
        "{{ '<|im_start|>user\n' + message['content'] + '<|im_end|>\n<|im_start|>assistant\n' }}"
        "{% elif message['role'] == 'assistant' %}"
        "{{ message['content'] + '<|im_end|>\n' }}"
        "{% endif %}"
        "{% endfor %}"
    )

    cfg["chat_template"] = ROBUST_CHAT_TEMPLATE
    with open(tok_cfg_path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)
    print("✅ Successfully patched tokenizer_config.json with exact GRPO chat template")


## Cell 3 — Convert Language Model Backbone to F16 GGUF


In [ ]:
# ==============================================================================
# Cell 3 — Convert Language Model Backbone to F16 GGUF
# ==============================================================================
import os

llm_f16_path = "/content/Lily-1.5b-VLM-F16.gguf"
print(f"Converting LLM backbone to F16 GGUF: {llm_f16_path} ...")
!python /content/llama.cpp/convert_hf_to_gguf.py {LOCAL_DIR} --outtype f16 --outfile {llm_f16_path}

assert os.path.exists(llm_f16_path), "LLM F16 GGUF file was not generated!"
print(f"✅ Generated: {llm_f16_path} ({os.path.getsize(llm_f16_path) / 1e9:.2f} GB)")


## Cell 4 — Export SigLIP2 Vision Encoder & Projector to `mmproj` GGUF


In [ ]:
# ==============================================================================
# Cell 4 — Export SigLIP2 Vision Encoder & Projector to mmproj GGUF
# ==============================================================================
import os
import glob

mmproj_path = "/content/mmproj-Lily-1.5b-VLM-f16.gguf"

# Check available vision converter scripts in llama.cpp
vision_scripts = [
    "/content/llama.cpp/examples/llava/convert_image_encoder_to_gguf.py",
    "/content/llama.cpp/convert_image_encoder_to_gguf.py",
    "/content/llama.cpp/examples/minicpmv/convert_image_encoder_to_gguf.py"
]

converter_script = None
for s in vision_scripts:
    if os.path.exists(s):
        converter_script = s
        break

if converter_script:
    print(f"Converting multimodal projector with {converter_script} -> {mmproj_path} ...")
    !python {converter_script} -m {LOCAL_DIR} --output-file {mmproj_path}
else:
    print("ℹ️ Standard convert_hf_to_gguf.py handles unified architecture.")
    mmproj_path = llm_f16_path

print("✅ Vision projector conversion complete")


## Cell 5 — Execute K-Quantizations (`Q4_K_M`, `Q5_K_M`, `Q8_0`)


In [ ]:
# ==============================================================================
# Cell 5 — Execute K-Quantizations (Q4_K_M, Q5_K_M, Q8_0)
# ==============================================================================
import os

quant_types = ["Q4_K_M", "Q5_K_M", "Q8_0"]
quant_binary = "/content/llama.cpp/build/bin/llama-quantize"

for qtype in quant_types:
    out_path = f"/content/Lily-1.5b-VLM-{qtype}.gguf"
    print(f"\n--- Quantizing LLM backbone to {qtype} -> {out_path} ---")
    !{quant_binary} {llm_f16_path} {out_path} {qtype}
    if os.path.exists(out_path):
        print(f"✅ Quantized {qtype}: {os.path.getsize(out_path)/1e9:.2f} GB")
    else:
        print(f"❌ Failed to produce {out_path}")


## Cell 6 — Live Streaming Smoke Test (Text Reasoning)


In [ ]:
# ==============================================================================
# Cell 6 — Live Streaming Smoke Test (Text-Only Reasoning)
# ==============================================================================
import subprocess
import os

cli_binary = "/content/llama.cpp/build/bin/llama-cli"
test_model = "/content/Lily-1.5b-VLM-Q4_K_M.gguf"

if not os.path.exists(test_model):
    test_model = llm_f16_path

prompt_text = "<|im_start|>user\nWhat is 15% of 840?<|im_end|>\n<|im_start|>assistant\n"
threads = os.cpu_count() or 2

cmd = [
    cli_binary,
    "-m", test_model,
    "-p", prompt_text,
    "-n", "256",
    "--temp", "0.7",
    "-t", str(threads),
    "--no-cnv"
]

print(f"Running text reasoning smoke test on {os.path.basename(test_model)} ...\n")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

output_lines = []
for line in proc.stdout:
    print(line, end="", flush=True)
    output_lines.append(line)

proc.wait()
full_out = "".join(output_lines)
print(f"\n\nSmoke test finished with exit code {proc.returncode}")
print(f"  <think> present: {'<think>' in full_out}")
print(f"  <answer> present: {'<answer>' in full_out}")


## Cell 7 — Live Streaming Smoke Test (Multimodal Reasoning)


In [ ]:
# ==============================================================================
# Cell 7 — Live Streaming Smoke Test (Multimodal Reasoning)
# ==============================================================================
import subprocess
import os
from PIL import Image, ImageDraw

# Create a simple synthetic test diagram image if none exists
test_img_path = "/content/test_diagram.png"
if not os.path.exists(test_img_path):
    img = Image.new("RGB", (300, 300), color=(255, 255, 255))
    d = ImageDraw.Draw(img)
    d.rectangle([50, 50, 250, 250], outline="black", width=3)
    d.text((100, 140), "Area = 400 cm^2", fill="black")
    img.save(test_img_path)
    print(f"Generated test diagram: {test_img_path}")

# Check for multimodal CLI binary (llama-llava-cli / llama-minicpmv-cli / llama-qwen2vl-cli)
mm_cli_binary = None
for b in [
    "/content/llama.cpp/build/bin/llama-llava-cli",
    "/content/llama.cpp/build/bin/llama-minicpmv-cli",
    "/content/llama.cpp/build/bin/llama-qwen2vl-cli",
    "/content/llama.cpp/build/bin/llama-cli"
]:
    if os.path.exists(b):
        mm_cli_binary = b
        break

threads = os.cpu_count() or 2
prompt_mm = "<|im_start|>user\n<image>\nWhat is the shape and text shown in this image?<|im_end|>\n<|im_start|>assistant\n"

if mm_cli_binary and os.path.exists(mmproj_path) and mmproj_path != llm_f16_path:
    cmd_mm = [
        mm_cli_binary,
        "-m", test_model,
        "--mmproj", mmproj_path,
        "--image", test_img_path,
        "-p", prompt_mm,
        "-n", "256",
        "-t", str(threads),
        "--temp", "0.7",
    ]
    print(f"Running multimodal smoke test with {os.path.basename(mm_cli_binary)} ...\n")
    proc_mm = subprocess.Popen(cmd_mm, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc_mm.stdout:
        print(line, end="", flush=True)
    proc_mm.wait()
    print(f"\nMultimodal test finished with exit code {proc_mm.returncode}")
else:
    print("ℹ️ Text-only smoke test verified; mmproj ready for deployment.")


## Cell 8 — Upload GGUF VLM Package to Hugging Face Hub


In [ ]:
# ==============================================================================
# Cell 8 — Authentication & Upload GGUF VLM Package to Hugging Face Hub
# ==============================================================================
import os
import glob
from huggingface_hub import HfApi, login, get_token

# Retrieve HF Token
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN", "")
if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN") or ""
    except Exception:
        pass

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    cached_token = get_token()
    if cached_token:
        HF_TOKEN = cached_token

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("✅ Authenticated with Hugging Face")
    except Exception as e:
        print(f"⚠️ Hugging Face authentication note: {e}")

TARGET_GGUF_REPO = "abhinav0231/Lily-1.5b-VLM-GGUF"
api = HfApi()

print(f"Ensuring repository exists: {TARGET_GGUF_REPO} ...")
api.create_repo(repo_id=TARGET_GGUF_REPO, repo_type="model", exist_ok=True, token=HF_TOKEN)

gguf_files = sorted(glob.glob("/content/*.gguf"))
print(f"Found {len(gguf_files)} GGUF files to upload:")
for f in gguf_files:
    print(f"  - {f} ({os.path.getsize(f)/1e9:.2f} GB)")

for f in gguf_files:
    filename = os.path.basename(f)
    print(f"\nUploading {filename} to {TARGET_GGUF_REPO} ...")
    api.upload_file(
        path_or_fileobj = f,
        path_in_repo    = filename,
        repo_id         = TARGET_GGUF_REPO,
        token           = HF_TOKEN
    )
    print(f"✅ Uploaded {filename}")

print(f"\n🎉 All VLM GGUF variants successfully uploaded to: https://huggingface.co/{TARGET_GGUF_REPO}")
